# 07 · Event backtest: long additions and short deletions into the index trade date

This notebook backtests the pre-event index effect directly from the event register, without any forecasting model. Every main addition is bought and every main deletion is sold short a fixed number of sessions `k` before its index trade date, and the position is closed at the close of the index trade date, when index funds trade. Only U.S. common stocks are traded: no derivatives and no ETFs.

**Design, fixed before computing any return**

| Element | Choice |
|---|---|
| Events | Main additions and main (discretionary) deletions from notebook 01, as in notebook 02 |
| Entry | Close of session `−k` relative to the index trade date (day 0). Main case `k = 5`; variants `k = 3` and `k = 1` |
| Exit | Close of day 0 |
| Samples | 1976–2022 and the sub-periods 1976–89, 1990–99, 2000–09, 2010–22, with a test of 2010–22 against 1990–99. 2023–2025 is a provisional robustness check |
| Per-trade results | Raw return and return minus the S&P 500, t-statistics clustered by effective month |
| Calendar-time results | Daily equal-weighted legs, cash at the risk-free rate when no position is open, compounded to months. FF5 + momentum alpha with Newey–West (12 lags) |
| Costs | Quoted half-spread at entry and at exit (or 0.2/0.5/1.0% per side); borrow fee 0.5/1/2% a year on shorts for the days held |

**What the entry assumes.** The register has no verified announcement dates for 1976–2022. Entering `k` sessions before the trade date assumes that the change was public by then. Since October 1989, S&P has usually announced changes about five sessions ahead, so `k = 5` is the main case and `k = 3` and `k = 1` are more conservative. **Before October 1989**, S&P announced changes after the close and they took effect at the next open. A position opened before day 0 therefore uses information that was not public, and the 1976–1989 results are not tradable. They are reported only for comparison.

**Upper bound, not a strategy.** The same trade with `k = 60` corresponds to the [−60, 0] CAR. Nobody could know the events 60 sessions ahead, so it is shown only as the perfect-foresight ceiling.

In [ ]:
from pathlib import Path

from IPython.utils.capture import capture_output
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import statsmodels.api as sm

inventory_path = Path("01_inventory.ipynb")
if not inventory_path.is_file():
    inventory_path = Path("notebooks") / "01_inventory.ipynb"
with capture_output(stdout=False, stderr=False):
    get_ipython().run_line_magic("run", f'"{inventory_path.resolve().as_posix()}"')

main_k = 5
k_values = [5, 3, 1]
foresight_k = 60
fallback_half_spread = 0.005
newey_west_lags = 12
sub_periods = ["1976–1989", "1990–1999", "2000–2009", "2010–2022"]
overall_period = "1976–2022"
recent_period = "2023–2025 (provisional)"

specification_log = []


def log_specifications(table_name, t_values):
    t_values = pd.Series(t_values).dropna()
    specification_log.append({"table": table_name, "estimates": len(t_values),
                              "|t| > 2": int(t_values.abs().gt(2).sum()), "|t| > 3": int(t_values.abs().gt(3).sum())})


def period_of(dates):
    years = dates.dt.year
    return pd.Series(np.select(
        [years.le(1989), years.le(1999), years.le(2009), years.le(2022)], sub_periods, default=recent_period
    ), index=dates.index)


samples = {
    "additions": additions_register.loc[
        (additions_register["main_sample"] | additions_register["sample"].eq("robustness_2023_2025"))
        & additions_register["permno"].notna() & ~additions_register["flag_no_prior_crsp_3m"]],
    "deletions": deletions_register.loc[
        (deletions_register["main_sample"] | deletions_register["sample"].eq("robustness_2023_2025"))
        & deletions_register["permno"].notna() & ~deletions_register["flag_delisting_related"]],
}
events = pd.concat([
    frame[["permno", "effective_date", "index_trade_date", "sample"]].assign(leg=leg)
    for leg, frame in samples.items()
], ignore_index=True).astype({"permno": "int64"})
events["period"] = period_of(events["effective_date"])
events["cluster_month"] = events["effective_date"].dt.to_period("M")
display(events.groupby(["leg", "period"]).size().unstack("leg"))

## Daily data

Daily returns and quotes of the event securities, from the Parquet copy of the daily CRSP file after checking its row count (as in notebook 02). The benchmark is the S&P 500 daily total return.

In [ ]:
expected_daily_rows = 92_586_611
daily_dataset = ds.dataset(data_dir / "parquet" / "crsp_daily", format="parquet")
if daily_dataset.count_rows() != expected_daily_rows:
    raise ValueError("The Parquet copy of the daily file does not match the CSV row count.")
daily = daily_dataset.to_table(
    columns=["permno", "dlycaldt", "dlydelflg", "dlyret", "dlybid", "dlyask"],
    filter=ds.field("permno").isin(sorted(events["permno"].unique())),
).to_pandas()
daily["date"] = pd.to_datetime(daily.pop("dlycaldt"))
daily = daily.drop_duplicates()
conflicting = daily.duplicated(["permno", "date"], keep=False)
daily.loc[conflicting, ["dlyret", "dlybid", "dlyask"]] = np.nan
daily = daily.drop_duplicates(["permno", "date"])
daily["session"] = daily["date"].map(session_of)
daily = daily.dropna(subset=["session"]).astype({"session": "int64"}).sort_values(["permno", "session"], ignore_index=True)

valid_quote = daily["dlybid"].gt(0) & daily["dlyask"].gt(daily["dlybid"])
daily["half_spread"] = ((daily["dlyask"] - daily["dlybid"]) / (daily["dlyask"] + daily["dlybid"])).where(valid_quote)
daily.loc[daily["half_spread"].ge(0.10), "half_spread"] = np.nan   # relative spread of 20% or more is treated as an error

benchmark = pd.read_csv(paths["index_daily"], usecols=["dlycaldt", "sprtrn"], parse_dates=["dlycaldt"])
benchmark = benchmark.assign(session=benchmark["dlycaldt"].map(session_of)).dropna(subset=["session"])
sp_return = benchmark.set_index(benchmark["session"].astype("int64"))["sprtrn"]

monthly_factors = read_monthly_factors(
    data_dir / "F-F_Research_Data_5_Factors_2x3_CSV.zip", ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "RF"]
).join(read_monthly_factors(data_dir / "F-F_Momentum_Factor_CSV.zip", ["Mom"]), validate="one_to_one")
factor_columns = ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"]
session_month = pd.Series(trading_calendar.to_period("M"), index=np.arange(len(trading_calendar)))
sessions_per_month = session_month.value_counts()
daily_rf = (monthly_factors["RF"].reindex(session_month.to_numpy()).to_numpy()
            / sessions_per_month.reindex(session_month.to_numpy()).to_numpy())
daily_rf = pd.Series(daily_rf, index=session_month.index)
display(pd.Series({"daily rows for event securities": len(daily),
                   "rows with a usable quote": int(daily["half_spread"].notna().sum())}, name="daily data"))

## Trades

A trade is complete when the security has a return on every session from `−k+1` to 0 and is listed at the entry close. Incomplete trades are dropped and counted. The trade's cost uses the quoted half-spread at entry and at exit. A missing quote is replaced by the median quoted half-spread of that year's trades, and by 0.5% when the year has none. The return is reported per trade, in percent, for the `k + 0` sessions held.

In [ ]:
by_permno = {permno: frame for permno, frame in daily.groupby("permno")}


def build_trades(k):
    rows = []
    for event in events.itertuples():
        day0 = session_of[event.index_trade_date]
        frame = by_permno.get(event.permno)
        record = {"leg": event.leg, "permno": event.permno, "effective_date": event.effective_date,
                  "period": event.period, "cluster_month": event.cluster_month, "entry": day0 - k, "exit": day0}
        if frame is None:
            rows.append({**record, "complete": False}); continue
        held = frame.loc[frame["session"].between(day0 - k + 1, day0)]
        listed_at_entry = frame["session"].le(day0 - k).any()
        complete = listed_at_entry and len(held) == k and held["dlyret"].notna().all()
        record["complete"] = complete
        if complete:
            record["ret"] = np.prod(1 + held["dlyret"].to_numpy()) - 1
            record["sp_ret"] = np.prod(1 + sp_return.reindex(held["session"]).to_numpy()) - 1
            quotes = frame.set_index("session")["half_spread"]
            record["hs_entry"] = quotes.get(day0 - k, np.nan)
            record["hs_exit"] = quotes.get(day0, np.nan)
        rows.append(record)
    trades = pd.DataFrame(rows)
    year = trades["effective_date"].dt.year
    year_median = pd.concat([trades["hs_entry"], trades["hs_exit"]]).groupby(pd.concat([year, year])).median()
    for column in ["hs_entry", "hs_exit"]:
        trades[f"{column}_quoted"] = trades[column].notna()
        trades[column] = trades[column].fillna(year.map(year_median)).fillna(fallback_half_spread)
    sign = np.where(trades["leg"].eq("additions"), 1.0, -1.0)
    trades["trade_ret"] = sign * trades["ret"]                       # return to the position: short deletions
    trades["trade_adjusted"] = sign * (trades["ret"] - trades["sp_ret"])
    trades["cost_estimated"] = trades["hs_entry"] + trades["hs_exit"]
    trades["borrow_days"] = np.where(trades["leg"].eq("deletions"), k, 0)
    return trades


trades = {k: build_trades(k) for k in [*k_values, foresight_k]}
display(pd.DataFrame({
    f"k = {k}": {
        "events": len(frame), "complete trades": int(frame["complete"].sum()),
        "dropped (no return on a held session or not listed at entry)": int((~frame["complete"]).sum()),
        "trades with a quote at entry, 1993–2022": frame.loc[frame["complete"] & frame["effective_date"].dt.year.between(1993, 2022), "hs_entry_quoted"].mean(),
    } for k, frame in trades.items()
}).round(3))

## Per-trade returns

Mean return per trade in percent, for the leg's position: buying additions, short-selling deletions. A positive number is a gain for the strategy. *Adjusted* subtracts the S&P 500 return over the same sessions. *Net* subtracts the estimated round-trip cost and, for shorts, a 1% a year borrow fee for the days held. The t-statistics are clustered by effective month.

In [ ]:
def clustered_t(values, clusters):
    if len(values) < 3:
        return np.nan
    fit = sm.OLS(np.asarray(values, float), np.ones(len(values))).fit(
        cov_type="cluster", cov_kwds={"groups": pd.factorize(clusters)[0]})
    return fit.tvalues[0]


def net_return(frame, per_side=None, borrow=0.01):
    cost = frame["cost_estimated"] if per_side is None else 2 * per_side
    return frame["trade_ret"] - cost - borrow * frame["borrow_days"] / 252


def trade_summary(frame, column_values):
    rows = []
    for leg in ["additions", "deletions"]:
        for period in [overall_period, *sub_periods, recent_period]:
            sample = frame.loc[frame["complete"] & frame["leg"].eq(leg)]
            sample = sample.loc[sample["period"].ne(recent_period)] if period == overall_period else sample.loc[sample["period"].eq(period)]
            row = {"leg": leg, "period": period, "trades": len(sample)}
            for name, values in column_values(sample).items():
                row[f"{name}_pct"] = 100 * values.mean()
                row[f"{name}_t"] = clustered_t(values, sample["cluster_month"])
            row["hit_rate"] = sample["trade_ret"].gt(0).mean()
            rows.append(row)
    return pd.DataFrame(rows).set_index(["leg", "period"])


def main_columns(sample):
    return {"raw": sample["trade_ret"], "adjusted": sample["trade_adjusted"], "net": net_return(sample)}


trade_tables = {k: trade_summary(trades[k], main_columns) for k in k_values}
for k in k_values:
    log_specifications(f"Per-trade returns, k = {k}", trade_tables[k].filter(like="_t").stack())
print(f"Main case, k = {main_k}: entry 5 sessions before the index trade date")
display(trade_tables[main_k].round(3))

In [ ]:
print("Sensitivity to the entry session: mean raw and net return per trade (%), t in brackets")
sensitivity = pd.concat({f"k = {k}": trade_tables[k][["trades", "raw_pct", "raw_t", "net_pct", "net_t"]] for k in k_values}, axis=1)
display(sensitivity.round(2))

foresight = trade_summary(trades[foresight_k], lambda s: {"raw": s["trade_ret"], "adjusted": s["trade_adjusted"]})
print(f"Perfect-foresight ceiling, k = {foresight_k} (not tradable)")
display(foresight[["trades", "raw_pct", "raw_t", "adjusted_pct", "adjusted_t"]].round(2))

### Has the trade weakened? 2010–2022 against 1990–1999

Difference in mean per-trade return (2010–2022 minus 1990–1999), regression on a period dummy with standard errors clustered by effective month.

In [ ]:
rows = []
for k in k_values:
    for leg in ["additions", "deletions"]:
        sample = trades[k].loc[trades[k]["complete"] & trades[k]["leg"].eq(leg) & trades[k]["period"].isin(["1990–1999", "2010–2022"])]
        for name, values in [("raw", sample["trade_ret"]), ("net", net_return(sample))]:
            late = sample["period"].eq("2010–2022").astype(float).to_numpy()
            fit = sm.OLS(values.to_numpy(), np.column_stack([np.ones(len(late)), late])).fit(
                cov_type="cluster", cov_kwds={"groups": pd.factorize(sample["cluster_month"])[0]})
            rows.append({"k": k, "leg": leg, "return": name, "1990–1999_pct": 100 * fit.params[0],
                         "2010–2022_pct": 100 * (fit.params[0] + fit.params[1]),
                         "difference_pct": 100 * fit.params[1], "t_stat": fit.tvalues[1]})
difference_table = pd.DataFrame(rows).set_index(["k", "leg", "return"])
log_specifications("Difference 2010–2022 vs 1990–1999", difference_table["t_stat"])
display(difference_table.round(3))

## Calendar-time portfolio

Per-trade means ignore how often capital is used. Here each leg is a fund: on each session it holds all open positions in equal weights, and holds cash at the risk-free rate when no position is open. Entry and exit costs are charged on the day they occur, in proportion to the position's weight, and the short leg pays the borrow fee while positions are open. Daily excess returns are compounded to months and regressed on FF5 + momentum.

*Long additions* is the long fund's excess return, *short deletions* is the risk-free rate minus the short fund's return, and the spread is the sum of the two, so a month with only one active leg counts that leg alone. Because the funds are in cash most of the time, the monthly alpha is much smaller than the per-trade return. The annualized figure is the more useful one.

In [ ]:
def calendar_returns(frame, per_side=None, borrow=0.0):
    complete = frame.loc[frame["complete"]]
    held = complete.loc[complete.index.repeat(complete["exit"] - complete["entry"])].copy()
    held["session"] = held["entry"] + held.groupby(level=0).cumcount() + 1
    returns = daily[["permno", "session", "dlyret"]]
    held = held.merge(returns, on=["permno", "session"], how="left")
    held["n_open"] = held.groupby(["leg", "session"])["permno"].transform("size")
    held["first_day"] = held["session"].eq(held["entry"] + 1)
    held["last_day"] = held["session"].eq(held["exit"])
    entry_cost = held["hs_entry"] if per_side is None else per_side
    exit_cost = held["hs_exit"] if per_side is None else per_side
    held["cost"] = (held["first_day"] * entry_cost + held["last_day"] * exit_cost) / held["n_open"]
    legs = held.groupby(["leg", "session"]).agg(ret=("dlyret", "mean"), cost=("cost", "sum")).reset_index()
    out = pd.DataFrame(index=session_month.index)
    out["rf"] = daily_rf
    for leg, sign in [("additions", 1.0), ("deletions", -1.0)]:
        part = legs.loc[legs["leg"].eq(leg)].set_index("session")
        excess = sign * (part["ret"] - out["rf"].reindex(part.index)) - part["cost"]
        if leg == "deletions":
            excess = excess - borrow / 252
        out[leg] = excess.reindex(out.index).fillna(0.0)
        out[f"{leg}_open"] = part["ret"].reindex(out.index).notna()
    out["spread"] = out["additions"] + out["deletions"]
    out["month"] = session_month
    monthly = out.groupby("month").agg(
        **{name: (name, lambda r: np.prod(1 + r) - 1) for name in ["additions", "deletions", "spread"]},
        additions_days=("additions_open", "sum"), deletions_days=("deletions_open", "sum"))
    return monthly


def estimate_alpha(series, lags=newey_west_lags):
    sample = pd.concat([series.rename("y"), monthly_factors[factor_columns]], axis=1, join="inner").dropna()
    calendar = pd.period_range(sample.index.min(), sample.index.max(), freq="M")
    sample = sample.reindex(calendar)
    observed = sample.notna().all(axis=1).to_numpy()
    y = sample.loc[observed, "y"].to_numpy()
    X = np.column_stack([np.ones(observed.sum()), sample.loc[observed, factor_columns]])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    n, p = X.shape
    scores = np.zeros((len(calendar), p))
    scores[observed] = X * (y - X @ beta)[:, None]
    S = scores.T @ scores
    for lag in range(1, min(lags, len(calendar) - 1) + 1):
        G = scores[lag:].T @ scores[:-lag]
        S += (1 - lag / (lags + 1)) * (G + G.T)
    bread = np.linalg.inv(X.T @ X)
    cov = bread @ S @ bread * n / (n - p)
    alpha, se = beta[0], np.sqrt(cov[0, 0])
    return {"months": n, "alpha_monthly_pct": 100 * alpha, "t_stat": alpha / se,
            "alpha_annual_pct": 100 * ((1 + alpha) ** 12 - 1), "market_beta": beta[1],
            "mean_annual_pct": 100 * ((1 + y.mean()) ** 12 - 1)}


calendar_periods = {
    overall_period: pd.period_range("1976-01", "2022-12", freq="M"),
    "1976–1989": pd.period_range("1976-01", "1989-12", freq="M"),
    "1990–1999": pd.period_range("1990-01", "1999-12", freq="M"),
    "2000–2009": pd.period_range("2000-01", "2009-12", freq="M"),
    "2010–2022": pd.period_range("2010-01", "2022-12", freq="M"),
    recent_period: pd.period_range("2023-01", "2025-12", freq="M"),
}
scenarios = {"gross": (None, 0.0, False), "net, estimated spread + 1% borrow": (None, 0.01, True),
             "net, 0.5% per side + 1% borrow": (0.005, 0.01, True), "net, estimated spread + 2% borrow": (None, 0.02, True)}
calendar = {}
for k in k_values:
    for scenario, (per_side, borrow, charge) in scenarios.items():
        if k != main_k and scenario not in ("gross", "net, estimated spread + 1% borrow"):
            continue
        frame = trades[k] if charge else trades[k].assign(hs_entry=0.0, hs_exit=0.0)
        calendar[(k, scenario)] = calendar_returns(frame, per_side if charge else 0.0, borrow)

rows = []
for (k, scenario), monthly in calendar.items():
    for strategy, column in [("Long additions", "additions"), ("Short deletions", "deletions"), ("Long-short", "spread")]:
        for period, months in calendar_periods.items():
            rows.append({"k": k, "scenario": scenario, "strategy": strategy, "period": period,
                         **estimate_alpha(monthly.loc[monthly.index.isin(months), column])})
calendar_table = pd.DataFrame(rows).set_index(["k", "scenario", "strategy", "period"])
log_specifications("Calendar-time alphas", calendar_table["t_stat"])
main_calendar = calendar[(main_k, "gross")]
display(pd.DataFrame({
    "months with an open addition": (main_calendar["additions_days"] > 0).groupby(main_calendar.index.year // 10 * 10).mean(),
    "months with an open deletion": (main_calendar["deletions_days"] > 0).groupby(main_calendar.index.year // 10 * 10).mean(),
}).round(2).rename_axis("decade"))
with pd.option_context("display.max_rows", 200, "display.max_columns", 50, "display.width", 250):
    for k in k_values:
        print(f"k = {k}: monthly alpha (%), Newey–West t, and compounded annual alpha (%)")
        display(calendar_table.loc[k][["alpha_monthly_pct", "t_stat", "alpha_annual_pct"]].unstack("scenario")
                .swaplevel(0, 1, axis=1).sort_index(axis=1, level=0, sort_remaining=False).round(2))

In [ ]:
spec_table = pd.DataFrame(specification_log)
display(spec_table)
display(spec_table[["estimates", "|t| > 2", "|t| > 3"]].sum().rename("total"))

## Summary

**Per trade, main case (entry 5 sessions before the index trade date, exit at its close).** Mean return per trade, t clustered by effective month:

| Period | Additions, gross | Additions, net | Deletions (short), gross | Deletions (short), net |
|---|---|---|---|---|
| 1976–1989 (not tradable) | 0.86% (2.91) | −0.13% (−0.46) | −1.34% (−1.11) | −2.55% (−1.92) |
| 1990–1999 | 7.64% (12.07) | 6.67% (10.54) | 10.73% (5.32) | 8.43% (4.15) |
| 2000–2009 | 4.29% (6.79) | 3.91% (6.47) | 13.05% (5.62) | 11.98% (5.27) |
| 2010–2022 | 0.78% (1.87) | 0.75% (1.79) | 1.14% (1.40) | 1.04% (1.28) |
| 2023–2025 (provisional) | 3.84% (2.88) | 3.79% (2.84) | −0.11% (−0.07) | −0.20% (−0.12) |

Compared with 1990–1999, the 2010–2022 return per trade is 6.9 points lower for additions (t −9.07) and 9.6 points lower for deletions (t −4.42).

**Calendar time, main case.** Long-short monthly alpha, gross: 18.5% (t 5.67) in 1990–1999, 14.3% (t 3.49) in 2000–2009, 1.5% (t 1.81) in 2010–2022. Net of estimated spreads and a 1% borrow fee: 15.5% (t 4.63), 12.7% (t 3.53), 1.4% (t 1.69). At 0.5% per side, 2010–2022 becomes −0.2% (t −0.25). The annualized figures (several hundred percent in the 1990s) assume that the whole fund rotates through one to three stocks a few days at a time. That is possible only with small capital, so they measure the strength of the effect, not an achievable return at scale.

**Entry timing matters, and the feasible entry is the latest one.** The only dated announcements in the data (19 additions in 2025) were made between 2 and 10 sessions before the trade date, with a quarter of them 2–3 sessions before. Entering 5 sessions ahead therefore uses information that was not yet public for part of the events. Entering 1 session ahead is feasible for all of them. With that entry, 2010–2022 gives +0.02% per addition (t 0.12) and +0.45% per deletion (t 1.39), and the net long-short alpha is 0.09% a month (t 0.24).

**Perfect foresight (60 sessions, not tradable)** would still have earned 11.7% per addition and 7.1% per deletion in 2010–2022. The price pressure exists, but it builds up before the change can be known.

**Limits.**
- Before October 1989 the trade is not feasible: changes were announced after the close and took effect at the next open.
- The main deletions exclude those followed by a delisting within 21 sessions. In 28 cases that delisting happened after the trade date, so the exclusion uses later information.
- Quotes are sparse before 1993, so early costs often use the 0.5% fallback.
- 2023–2025 uses provisional event lists.

**Specifications.** This notebook reports 264 estimates with a t-statistic, 145 of them with |t| > 2. Most of the significant ones are 1990–2009 rows and per-trade means.

**Conclusion.** Buying additions and shorting deletions into the index trade date was a very large and significant trade in 1990–2009, also after costs. In 2010–2022 it is small and not statistically different from zero, gross or net, with any entry from one to five sessions before the trade. With the one entry that is certainly feasible, it is essentially zero. The provisional 2023–2025 additions show a positive result for the 5-session entry, but not for the feasible 1-session entry.